<a href="https://colab.research.google.com/github/maiikhanhh/restaurant-menu-performance-sql/blob/main/restaurant_orders_sql_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **New Menu Performance Analysis (Q1 2023)**


## **1. Overview**
Taste of the World Cafe, a restaurant serving a diverse menu of international cuisine, launched a new menu at the start of the year 2023. This analysis uses the menu's first full quarter of order data (January-March 2023) to assess how it is performing before deciding on further menu investment, intended for the restaurant manager and menu development team.

Taste of the World Cafe is a fictitious restaurant used for analytical practice. The dataset and findings are illustrative rather than from an actual business. Data sourced from Maven Analytics.

**Objectives**

- Examine the new menu and the order data collected
- Identify which items and cuisines are performing well or poorly, and what they reveal about customer preferences
- Recommend which cuisines the café should prioritise for further menu development

**Business Question**

How does the new menu perform across items and cuisines over the first quarter, and what should the café prioritise for further development?

**Dataset**

Two tables used to support this analysis:
- `order_details`(12,234 order lines across 5,370 distinct orders): order_details_id, order_id, order_date, order_time, **item_id**
- `menu_items`(32 rows for 32 menu items): **menu_item_id**, item_name, category, price


## **2. Data Import and Preparation**

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving menu_items.csv to menu_items.csv
Saving order_details.csv to order_details.csv
Saving restaurant_db_data_dictionary.csv to restaurant_db_data_dictionary.csv


In [ ]:
import pandas as pd
import sqlite3

# Load datasets
menu_items = pd.read_csv("menu_items.csv")
order_details = pd.read_csv("order_details.csv")
data_dict = pd.read_csv("restaurant_db_data_dictionary.csv")


# The date and time fields would be converted into SQLite-friendly formats so they could be used for weekly, weekday, and hourly analysis
# Convert 'order_date' to datetime format and then format to 'YYYY-MM-DD'
order_details['order_date'] = pd.to_datetime(order_details['order_date'], format='%m/%d/%y').dt.strftime('%Y-%m-%d')

# Convert 'order_time' to datetime format and then format to 'HH:MM:SS'
order_details['order_time'] = pd.to_datetime(order_details['order_time'], format='%I:%M:%S %p').dt.strftime('%H:%M:%S')

# A glance at order_details table
order_details.head()

,order_details_id,order_id,order_date,order_time,item_id
0,1,1,2023-01-01,11:38:36,109.0
1,2,2,2023-01-01,11:57:40,108.0
2,3,2,2023-01-01,11:57:40,124.0
3,4,2,2023-01-01,11:57:40,117.0
4,5,2,2023-01-01,11:57:40,129.0


In `order_details` table, `order_date` was originally stored in `MM/DD/YY` format and `order_time` in
`HH:MM:SS AM/PM`, then converted to SQLite-compatible formats using pandas before loading into SQLite. This conversion was necessary to enable weekly, weekday, and hourly aggregations in the analysis.


In [ ]:
# A glance at menu_items table
menu_items.head()

,menu_item_id,item_name,category,price
0,101,Hamburger,American,12.95
1,102,Cheeseburger,American,13.95
2,103,Hot Dog,American,9.00
3,104,Veggie Burger,American,10.50
4,105,Mac & Cheese,American,7.00


In [ ]:
# Variable description
data_dict

,Table,Field,Description
0,menu_items,menu_item_id,Unique ID of a menu item
1,menu_items,item_name,Name of a menu item
2,menu_items,category,Category or type of cuisine of the menu item
3,menu_items,price,Price of the menu item (US Dollars $)
4,order_details,order_details_id,Unique ID of an item in an order
5,order_details,order_id,ID of an order
6,order_details,order_date,Date an order was put in (MM/DD/YY)
7,order_details,order_time,Time an order was put in (HH:MM:SS AM/PM)
8,order_details,item_id,Matches the menu_item_id in the menu_items table


### **Checking for Nulls and Duplicates**

In [ ]:
# Check for null values in menu_items
print("Null values in menu_items:")
display(menu_items.isnull().sum())

# Check for duplicate rows in menu_items
print("\nDuplicate rows in menu_items:")
display(menu_items.duplicated().sum())

Null values in menu_items:


,0
menu_item_id,0
item_name,0
category,0
price,0



Duplicate rows in menu_items:


np.int64(0)

In [ ]:
# Check for null values in order_details
print("Null values in order_details:")
display(order_details.isnull().sum())

# Check for duplicate rows in order_details
print("\nDuplicate rows in order_details:")
display(order_details.duplicated().sum())

Null values in order_details:


,0
order_details_id,0
order_id,0
order_date,0
order_time,0
item_id,137



Duplicate rows in order_details:


np.int64(0)

In [ ]:
# Remove rows where item_id is null
order_details = order_details.dropna(subset=['item_id'])

No duplicate records were found in either table.

No null values were found in `menu_items`. In `order_details`, 137 null values were identified in the `item_id` field, all other fields in the affected rows were complete. Since `item_id` is the join key to
`menu_items`, these rows cannot contribute to any item or revenue analysis and should be removed.

As the result, 137 rows were removed, reducing `order_details` from 12,234 rows to 12,097 rows (approximately 1.12%), representing a small proportion of data loss that is unlikely to substantially impact the analysis.

It is worth noting that these unknown nulls might be due to data entry errors or system logging issues rather than zero-item orders.


In [ ]:
conn = sqlite3.connect("restaurant_orders.db")

menu_items.to_sql("menu_items", conn, if_exists="replace", index=False)
order_details.to_sql("order_details", conn, if_exists="replace", index=False)

12097

In [ ]:
query = """
SELECT *
FROM menu_items
"""

pd.read_sql(query, conn)


,menu_item_id,item_name,category,price
0,101,Hamburger,American,12.95
1,102,Cheeseburger,American,13.95
2,103,Hot Dog,American,9.00
3,104,Veggie Burger,American,10.50
4,105,Mac & Cheese,American,7.00
5,106,French Fries,American,7.00
6,107,Orange Chicken,Asian,16.50
7,108,Tofu Pad Thai,Asian,14.50
8,109,Korean Beef Bowl,Asian,17.95
9,110,Pork Ramen,Asian,17.95


In [ ]:
query = """
SELECT *
FROM order_details
LIMIT 10
"""

pd.read_sql(query, conn)

,order_details_id,order_id,order_date,order_time,item_id
0,1,1,2023-01-01,11:38:36,109.0
1,2,2,2023-01-01,11:57:40,108.0
2,3,2,2023-01-01,11:57:40,124.0
3,4,2,2023-01-01,11:57:40,117.0
4,5,2,2023-01-01,11:57:40,129.0
5,6,2,2023-01-01,11:57:40,106.0
6,7,3,2023-01-01,12:12:28,117.0
7,8,3,2023-01-01,12:12:28,119.0
8,9,4,2023-01-01,12:16:31,117.0
9,10,5,2023-01-01,12:21:30,117.0


##  **3. Data Analysis**



### **Demand and Timing Pattern**

**Business question**:
"When does the restaurant generate the most orders and revenue, and is demand growing across the quarter?"

In [ ]:
# Average Order Value (AOV)
query = """
SELECT
    COUNT(DISTINCT o.order_id) AS num_orders,
    ROUND(SUM(m.price),2) AS revenue,
    ROUND(SUM(m.price)/COUNT(DISTINCT o.order_id),2) AS avg_order_value
FROM order_details AS o
JOIN menu_items AS m
    ON o.item_id = m.menu_item_id
"""
pd.read_sql(query, conn)

,num_orders,revenue,avg_order_value
0,5343,159217.9,29.8


In [ ]:
# Revenue trend over the quarter by week
query = """
SELECT
    STRFTIME('%W',o.order_date) as week,
    COUNT(DISTINCT o.order_id) AS num_orders,
    ROUND(SUM(m.price),2) AS revenue
FROM order_details AS o
JOIN menu_items AS m
    ON o.item_id = m.menu_item_id
GROUP BY 1
ORDER BY 1 ;
"""
pd.read_sql(query, conn)

,week,num_orders,revenue
0,00,68,2091.60
1,01,430,12762.30
2,02,405,11251.70
3,03,410,12355.90
4,04,403,11604.00
5,05,448,13562.30
6,06,416,12419.65
7,07,408,12302.35
8,08,393,12037.50
9,09,412,12836.65


In [ ]:
# Revenue by weekday
query = """
WITH daily_orders AS (
SELECT
    STRFTIME('%d/%m/%Y',o.order_date) as date,
    CASE STRFTIME('%w', o.order_date)
        WHEN '0' THEN 'Sun'
        WHEN '1' THEN 'Mon'
        WHEN '2' THEN 'Tues'
        WHEN '3' THEN 'Wed'
        WHEN '4' THEN 'Thu'
        WHEN '5' THEN 'Fri'
        WHEN '6' THEN 'Sat'
    END AS weekday,
    COUNT(DISTINCT o.order_id) AS num_orders,
    ROUND(SUM(m.price),2) AS revenue
FROM order_details AS o
JOIN menu_items AS m
    ON o.item_id = m.menu_item_id
GROUP BY date)
SELECT
    weekday,
    SUM(num_orders) AS total_orders,
    ROUND(SUM(revenue),2) AS total_revenue,
    ROUND(SUM(revenue)/SUM(num_orders),2) AS avg_order_value
FROM daily_orders
GROUP BY weekday
ORDER BY 3 DESC;

"""
pd.read_sql(query, conn)

,weekday,total_orders,total_revenue,avg_order_value
0,Mon,881,26007.45,29.52
1,Fri,784,23707.95,30.24
2,Tues,759,23356.00,30.77
3,Sun,792,23226.45,29.33
4,Thu,739,21846.85,29.56
5,Sat,708,21170.90,29.90
6,Wed,680,19902.30,29.27


In [ ]:
# Revenue by hour
query = """
SELECT
    STRFTIME('%H', order_time) AS order_hour,
    COUNT(DISTINCT order_id) AS num_orders,
    ROUND(SUM(m.price),2) AS revenue,
    ROUND(SUM(m.price)/COUNT(DISTINCT o.order_id),2) AS avg_order_value
FROM order_details AS o
JOIN menu_items AS m
    ON o.item_id = m.menu_item_id
GROUP BY order_hour
ORDER BY 3 desc;
"""

pd.read_sql_query(query, conn)

,order_hour,num_orders,revenue,avg_order_value
0,12,644,21718.40,33.72
1,13,590,20640.25,34.98
2,17,618,17869.50,28.92
3,18,595,16861.50,28.34
4,19,497,14172.60,28.52
5,16,479,13711.65,28.63
6,14,423,12615.70,29.82
7,20,411,11678.60,28.42
8,15,354,9803.90,27.69
9,11,286,8122.50,28.40


Looking at these numbers, weekly revenue holds steady between roughly \$11,200 - \$13,600 for 12 of the 14 weeks, with no clear upward trend. This means the menu has not yet generated organic growth across the quarter

Monday and Friday seem to outperform the weekend in terms of sales, while  average order value stays consistent across all days (~ \$29–31). The consistency points to a volume effect: revenue swings by day come from how many customers walk in, not how much they spend once they are there. However, Monday being the best day rather than weekend is unusual for a restaurant and worth investigating further; it may reflect a local pattern (a nearby office, a recurring booking) rather than anything the menu itself is doing.

Lunch time (12-1pm) is both the busiest period of the day and the highest-value one, with an AOV of approximately \$34 - roughly 20% higher than dinner (5–7pm), the second busiest time. This shows that the new menu is most lucrative during lunch.

**Insight:**
- The new menu looks stabilised but not actually growing, which is likely because the new menu launch generated initial curiosity but no active promotions have been done for a more sustainable growth.
- Monday leads on revenue through volume alone but average order value is consistent across the week, meaning the gap comes entirely from order volume, not spend per customer. However, based on the dataset alone, the cause is not identifiable to explain the matter without further details on local context and extra information.
- Lunch is the highest-value time with higher average spending per order than dinner, which means customer tend to place more complete/full meal orders (starters or sides plus main). This can be viewed as entailing higher risk, given that service failure or a stockout may incur higher costs than the same at dinner.



### **Menu and Category Performance**

**Business question**: "Which menu items and cuisine categories are driving revenue, and which are underperforming?"

In [ ]:
# Revenue by Menu items
query = """
SELECT m.item_name,
    m.category,
    m.price,
    COUNT(o.item_id) AS item_sold,
    ROUND(SUM(m.price),2) AS revenue
FROM order_details AS o
JOIN menu_items AS m
    ON o.item_id = m.menu_item_id
GROUP BY 1
ORDER BY revenue DESC;
"""

pd.read_sql_query(query, conn)



,item_name,category,price,item_sold,revenue
0,Korean Beef Bowl,Asian,17.95,588,10554.60
1,Spaghetti & Meatballs,Italian,17.95,470,8436.50
2,Tofu Pad Thai,Asian,14.50,562,8149.00
3,Cheeseburger,American,13.95,583,8132.85
4,Hamburger,American,12.95,622,8054.90
5,Orange Chicken,Asian,16.50,456,7524.00
6,Eggplant Parmesan,Italian,16.95,420,7119.00
7,Steak Torta,Mexican,13.95,489,6821.55
8,Chicken Parmesan,Italian,17.95,364,6533.80
9,Pork Ramen,Asian,17.95,360,6462.00


In [ ]:
# Revenue by Cuisine Category
query = """
WITH total_revenue AS (
SELECT
    COUNT(o.item_id) AS items_sold,
    ROUND(SUM(m.price), 2) AS revenue
FROM order_details AS o
JOIN menu_items AS m
    ON o.item_id = m.menu_item_id
)
SELECT
    m.category,
    COUNT(o.item_id) AS items_sold,
    ROUND((CAST(COUNT(o.item_id) AS REAL)/t.items_sold)*100, 2) AS items_pct,
    ROUND(SUM(m.price), 2) AS revenue,
    ROUND((SUM(m.price)/t.revenue)*100, 2) AS revenue_pct,
    ROUND((SUM(m.price)/COUNT(o.item_id)),2) AS avg_price
FROM order_details AS o
JOIN menu_items AS m
    ON o.item_id = m.menu_item_id
CROSS JOIN total_revenue AS t
GROUP BY m.category
ORDER BY 4 DESC;
"""
pd.read_sql_query(query, conn)

,category,items_sold,items_pct,revenue,revenue_pct,avg_price
0,Italian,2948,24.37,49462.70,31.07,16.78
1,Asian,3470,28.68,46720.65,29.34,13.46
2,Mexican,2945,24.34,34796.80,21.85,11.82
3,American,2734,22.60,28237.75,17.74,10.33


Korean Beef Bowl leads revenue and also sits among top three best selling dishes. Chicken Tacos sits at the opposite end, with the lowest volume (123 items sold) and the lowest revenue of any dish on the menu.

At category level, Italian earns the most, nearly one third of the total from fewer items sold than Asian, the most popular cuisine by volume. With the roughly similar number of items sold compared to Italian, Mexican sells almost as many items as Italian but earns considerably less revenue. The gap comes down to price rather than popularity, since Mexican's average item price is the second-lowest of the four.

American underperforms on both volume and revenue of any category. Eventhough only around 7% lower in sales volume, American revenue share was only about 60% compared to Italian, indicating a significant difference in actual revenue.

**Insight:**
- Italian earns more because of price, not popularity like Asian, making it the strongest revenue contributor.
- American is the menu's weakest category on every dimension, a low-volume, low-priced, low-demand category.

### **High-spend Orders**

**Business question**: "What do the highest-spend orders tell about top customers' preference?"

In [ ]:
# Spending by Orders
query = """
With high_spending AS (
SELECT
    o.order_id,
    COUNT(o.item_id) AS items_ordered,
    ROUND(SUM(m.price),2) AS order_value
FROM order_details AS o
JOIN menu_items AS m
    ON o.item_id = m.menu_item_id
GROUP BY 1
ORDER BY 3 desc
LIMIT 10
)
SELECT
    m.item_name,
    m.category,
    m.price,
    COUNT(o.item_id) AS times_ordered,
    ROUND(m.price * COUNT(o.item_id), 2) AS revenue
FROM high_spending AS h
JOIN order_details AS o
    ON h.order_id = o.order_id
JOIN menu_items AS m
    ON o.item_id = m.menu_item_id
GROUP BY 2,1
ORDER BY 4 desc;
"""

pd.read_sql_query(query, conn)

,item_name,category,price,times_ordered,revenue
0,Eggplant Parmesan,Italian,16.95,10,169.50
1,Chips & Salsa,Mexican,7.00,9,63.00
2,Spaghetti & Meatballs,Italian,17.95,8,143.60
3,Korean Beef Bowl,Asian,17.95,7,125.65
4,Salmon Roll,Asian,14.95,7,104.65
5,Chicken Parmesan,Italian,17.95,7,125.65
6,Cheeseburger,American,13.95,6,83.70
7,Tofu Pad Thai,Asian,14.50,6,87.00
8,Spaghetti,Italian,14.50,6,87.00
9,Chicken Burrito,Mexican,12.95,6,77.70


Across the top 10 highest-spend orders, Italian dominates both in frequency and revenue contribution, appearing more than any other category and generating the highest revenue. Eggplant Parmesan is the most frequently chosen item in these orders with 10 appearances (averaging one per order), making it a consistent item of choice among high spenders.

Mexican ranks second by frequency, but this is weighted largely toward lower-priced items, limiting revenue contribution. This pattern again reinforces the category-level finding: Italian is where high-spending tables concentrate their spend, both across the full dataset and particularly among the top value orders.

**Insight:** High-spending orders consistently include Italian dishes, particularly Eggplant Parmesan appearing most often across the top 10 highest-spend orders. This indicates customer willingness is due to genuine preference and not affected by price, showing this is a considerable potential category for future menu development

##  **4. Recommendations**
- Prioritise Italian and Asian for menu refresh by adding a few new items in each category, remaining consistent with existing bestsellers. Italian for its strong revenue return and Asian for its leading sales volume, both are proven performers worth building on. In the following period, revenue share of these two categories should be continuously reviewed.
- Keep lunch service prepared above all else. The 12–1pm window is both the busiest and highest-value period of the day so ensure full menu availability
and adequate staffing from 11:30am–1:30pm.
- Reassess Mexican pricing. Mexican's average price is the second-lowest of the four cuisines, limiting its revenue contribution despite comparable volume to Italian. Introducing higher price on  select mains and evaluate customer response in the following quarter against the current Q1 baseline.
- Investigate American before any further development. American's weak performance on volume and revenue suggests conducting a review of order patterns of certain dishes to identify if this is a menu composition issue or a demand constraint.
- Chicken Tacos, with the least items sold and revenue made, should be considered replacing or removing from the menu, addressing both the underperformance and pricing gap simultaneously.
- Investigate the Monday and weekend pattern before acting on it. Conduct a brief local context review: nearby office
or residential density, recurring bookings, competitor trading patterns. If Monday's performance indicates reliable demand, formalise it with targeted promotions or adequate staffing. Also, if
Wednesday revenue consistently fails to cover operating costs, a one-day
closure is worth evaluating, though this requires cost data not available
in this analysis.



##  **5. Limitations**
- No pre-launch menu data, this performance is a first-quarter snapshot, no comparison on effectiveness
- Purely revenue figures, not profit as there is no cost data to evaluate the margin
- No customer ID or customer profile data for further insights, order-level patterns cannot be tied to individual or repeat customers
